# MIR — MobileNetV2 (Zero-Shot + Finetuning ArcFace)

Notebook équivalent à la version SupCon mais avec **ArcFace Loss** comme objectif de finetuning. Même méthodologie :
1. Évaluation zero-shot (Recall@K + mAP)
2. Finetuning avec **ArcFaceLoss** (margin angulaire) sur des embeddings normalisés
3. **Suivi de la dynamique d'entraînement** : courbes train/val loss + Val Recall@1 par epoch
4. Évaluation après finetuning
5. Comparaison Zero-Shot vs ArcFace Finetuned

Split stratifié train/val à 85/15 par classe pour pouvoir tracer une vraie courbe de validation loss.
Toutes les sauvegardes (modèles, embeddings, graphes, résultats JSON, historique) vont dans `MobileNetV2_ArcFace_training` sur le Drive.

In [ ]:
!pip install pytorch-metric-learning -q

In [ ]:
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
IMAGE_DIR    = os.path.join(PROJECT_ROOT, "dataset")
SAVE_DIR     = os.path.join(PROJECT_ROOT, "results", "MobileNetV2_ArcFace_training")
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 64   # MobileNetV2 est plus léger que DINOv2, on peut monter

print(f'Dataset : {IMAGE_DIR}')
print(f'Sauvegardes : {SAVE_DIR}')
print(f'GPU : {os.popen("nvidia-smi --query-gpu=name --format=csv,noheader").read().strip()}')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Utilisation de : {device}')

# Extraction du label depuis le nom de fichier
# Format attendu : 0_3_BMW_Serie5_474.jpg -> 'BMW_Serie5'
def get_label_from_filename(filename):
    name_without_ext = filename.split('.')[0]
    parts = name_without_ext.split('_')
    if len(parts) >= 4:
        return f'{parts[2]}_{parts[3]}'
    return 'Label_Inconnu'

## 1. MobileNetV2 Zero-Shot

Extraction des features pré-entraînées sur ImageNet sans aucun finetuning. Sert de baseline.

In [ ]:
class MobileNetV2FeatureExtractor(nn.Module):
    """MobileNetV2 sans la tête de classification. Sortie : 1280-dim."""
    def __init__(self):
        super().__init__()
        weights = models.MobileNet_V2_Weights.DEFAULT
        base = models.mobilenet_v2(weights=weights)
        self.features = base.features            # blocs convolutifs
        self.pool     = nn.AdaptiveAvgPool2d(1)  # global avg pool -> (B, 1280, 1, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return torch.flatten(x, 1)  # (B, 1280)


model_zs = MobileNetV2FeatureExtractor().to(device)
model_zs.eval()

n_params = sum(p.numel() for p in model_zs.parameters())
print(f'MobileNetV2 chargé. Paramètres : {n_params/1e6:.2f}M')
print('Dimension de sortie : 1280')

In [ ]:
class SimpleImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_filenames = [
            f for f in os.listdir(root_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))
        ]

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_filenames[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.image_filenames[idx]


# Transformations standard ImageNet pour MobileNetV2
transform_eval = transforms.Compose([
    transforms.CenterCrop(224),  # images déjà en 224px
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

dataset    = SimpleImageDataset(root_dir=IMAGE_DIR, transform=transform_eval)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=4)

print(f'Dataset chargé : {len(dataset)} images')

In [ ]:
all_embeddings = []
all_filenames  = []

print(f'Extraction des vecteurs pour {len(dataset)} images...')

with torch.no_grad():
    for images, filenames in tqdm(dataloader):
        images = images.to(device)
        embeddings = model_zs(images)
        embeddings = F.normalize(embeddings, p=2, dim=1)  
        all_embeddings.append(embeddings.cpu())
        all_filenames.extend(filenames)

database_tensor = torch.cat(all_embeddings, dim=0)

save_path = os.path.join(SAVE_DIR, 'MobileNetV2_zero_shot.pth')
torch.save({'embeddings': database_tensor, 'filenames': all_filenames}, save_path)

print(f'Base de données créée : {database_tensor.shape}')
print(f'Sauvegardée : {save_path}')

In [ ]:
db         = torch.load(os.path.join(SAVE_DIR, 'MobileNetV2_zero_shot.pth'))
embeddings = db['embeddings']
filenames  = db['filenames']
labels     = [get_label_from_filename(f) for f in filenames]

K_values    = [1, 5, 20, 50]
recalls     = {k: 0 for k in K_values}
ap_scores   = []
num_queries = len(embeddings)

print(f'Évaluation du Recall sur {num_queries} images...')

similarity_matrix = torch.mm(embeddings, embeddings.T)

for i in range(num_queries):
    query_label = labels[i]
    scores      = similarity_matrix[i].clone()
    scores[i]   = -1.0

    _, top_indices   = torch.topk(scores, max(K_values))
    retrieved_labels = [labels[idx.item()] for idx in top_indices]

    
    for k in K_values:
        if query_label in retrieved_labels[:k]:
            recalls[k] += 1

    
    num_relevant = labels.count(query_label) - 1
    if num_relevant == 0:
        continue

    hits, precision_sum = 0, 0.0
    for rank, lbl in enumerate(retrieved_labels, start=1):
        if lbl == query_label:
            hits += 1
            precision_sum += hits / rank
    ap_scores.append(precision_sum / num_relevant)

recall_scores = {k: recalls[k] / num_queries * 100 for k in K_values}
mAP           = sum(ap_scores) / len(ap_scores) * 100

print('\n--- Résultats Zero-Shot MobileNetV2 ---')
for k, v in recall_scores.items():
    print(f'Recall@{k:>2} : {v:.2f}%')
print(f'mAP        : {mAP:.2f}%')


plt.figure(figsize=(10, 6))
plt.plot(K_values, list(recall_scores.values()),
         marker='o', linestyle='-', color='#ff7f0e',
         linewidth=2.5, markersize=8, label='MobileNetV2')

for k, v in recall_scores.items():
    plt.annotate(f'{v:.1f}%', (k, v),
                 textcoords='offset points',
                 xytext=(0, 10), ha='center', fontsize=10)

plt.text(0.98, 0.05, f'mAP : {mAP:.2f}%',
         transform=plt.gca().transAxes, fontsize=11,
         ha='right', va='bottom',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.title('Courbe de Rappel (Recall@K) - MobileNetV2 zero shot',
          fontsize=14, fontweight='bold')
plt.xlabel('Rang K (Nombre de résultats affichés)', fontsize=12)
plt.ylabel('Taux de Rappel (%)', fontsize=12)
plt.xticks(K_values)
plt.ylim(0, 105)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()

graph_filename = 'recall_curve_MobileNetV2_zero_shot'
graph_path     = os.path.join(SAVE_DIR, f'{graph_filename}.png')
result_path    = os.path.join(SAVE_DIR, f'results_{graph_filename}.json')

plt.savefig(graph_path, dpi=300, bbox_inches='tight')

results = {
    'model':    graph_filename,
    'recall':   {f'@{k}': round(v, 2) for k, v in recall_scores.items()},
    'mAP':      round(mAP, 2),
    'n_images': num_queries,
}
with open(result_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Graphique sauvegardé : {graph_path}')
print(f'Résultats sauvegardés : {result_path}')
plt.show()

## 2. MobileNetV2 Finetuning — ArcFace

Backbone partiellement dégelé (2 derniers blocs) + tête de projection + **ArcFaceLoss**. ArcFace ajoute une marge angulaire `m` entre les classes dans l'espace cosinus :

$$ L = -\log\frac{e^{s\cdot\cos(\theta_{y_i}+m)}}{e^{s\cdot\cos(\theta_{y_i}+m)} + \sum_{j\neq y_i}e^{s\cdot\cos\theta_j}} $$

Avantages vs SupCon pour notre cas (46 classes, ~109 imgs/classe) :
- Marge explicite entre classes → embeddings plus discriminants pour la retrieval
- Pas besoin de batch size énorme ni de mining de paires
- Convergence plus stable

Les paramètres de la loss (matrice W des prototypes de classe) doivent être inclus dans l'optimizer.

In [ ]:
from pytorch_metric_learning import losses

In [ ]:
import random
from collections import defaultdict
from torch.utils.data import Subset

class LabeledImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_filenames = [
            f for f in os.listdir(root_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))
        ]
        all_labels    = [get_label_from_filename(f) for f in self.image_filenames]
        unique_labels = sorted(set(all_labels))
        self.label2idx = {l: i for i, l in enumerate(unique_labels)}
        self.targets   = [self.label2idx[l] for l in all_labels]

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_filenames[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.targets[idx]


# Augmentations pour l'entraînement
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4,
                           saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# Datasets train (avec aug) et val (sans aug) — mêmes indices internes,
# le split se fait sur les indices.
full_train_ds = LabeledImageDataset(root_dir=IMAGE_DIR, transform=train_transform)
full_eval_ds  = LabeledImageDataset(root_dir=IMAGE_DIR, transform=transform_eval)
NUM_CLASSES   = len(full_train_ds.label2idx)

# Split stratifié 85/15 par classe
VAL_RATIO = 0.15
random.seed(42)

by_class = defaultdict(list)
for idx, t in enumerate(full_train_ds.targets):
    by_class[t].append(idx)

train_indices, val_indices = [], []
for cls, idxs in by_class.items():
    random.shuffle(idxs)
    n_val = max(1, int(len(idxs) * VAL_RATIO))
    val_indices.extend(idxs[:n_val])
    train_indices.extend(idxs[n_val:])

# Subsets : train_subset utilise les transforms d'augmentation,
# val_subset utilise les transforms d'évaluation (pas d'aug)
train_subset = Subset(full_train_ds, train_indices)
val_subset   = Subset(full_eval_ds,  val_indices)

# Labels du split train pour configurer ArcFace
train_targets = [full_train_ds.targets[i] for i in train_indices]
val_targets   = [full_train_ds.targets[i] for i in val_indices]

print(f'Classes : {NUM_CLASSES}')
print(f'Train   : {len(train_subset)} images')
print(f'Val     : {len(val_subset)} images')
print(f'Total   : {len(train_subset) + len(val_subset)} images')


In [ ]:
class CarRetrievalMobileNetArcFace(nn.Module):
    """
    Pipeline ArcFace :
    MobileNetV2 features (1280) -> Linear -> BatchNorm -> embedding (256)
    L'embedding est passé à ArcFaceLoss qui le L2-normalise et applique
    la marge angulaire avec sa matrice de prototypes W (num_classes x 256).
    """
    def __init__(self, embed_dim=256):
        super().__init__()
        weights = models.MobileNet_V2_Weights.DEFAULT
        base    = models.mobilenet_v2(weights=weights)
        self.features = base.features
        self.pool     = nn.AdaptiveAvgPool2d(1)

        # Geler le backbone, sauf les 2 derniers blocs
        for p in self.features.parameters():
            p.requires_grad = False
        for p in self.features[-2:].parameters():
            p.requires_grad = True

        # Tête d'embedding (linear + BN très standard pour ArcFace)
        self.embedding = nn.Sequential(
            nn.Linear(1280, embed_dim),
            nn.BatchNorm1d(embed_dim),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)        # (B, 1280)
        return self.embedding(x)        # (B, embed_dim) — non normalisé


EMBED_DIM = 256
model = CarRetrievalMobileNetArcFace(embed_dim=EMBED_DIM).to(device)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Paramètres entraînables (backbone partiel + tête) : {n_trainable:,}')

In [ ]:
# ArcFaceLoss contient la matrice W des prototypes de classe et a ses propres paramètres.
# margin=28.6° et scale=64 sont les valeurs du papier original.
loss_fn = losses.ArcFaceLoss(
    num_classes=NUM_CLASSES,
    embedding_size=EMBED_DIM,
    margin=28.6,   # en degrés
    scale=64,
).to(device)

# Random shuffle suffit pour ArcFace (loss type classification avec marge)
train_loader = DataLoader(
    train_subset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    drop_last=True,
)

val_loader = DataLoader(
    val_subset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    drop_last=False,
)

# IMPORTANT : on optimise à la fois le modèle ET les paramètres de la loss (W).
# LR plus élevé pour les prototypes de classe (convention ArcFace).
optimizer = optim.AdamW(
    [
        {'params': [p for p in model.parameters() if p.requires_grad], 'lr': 1e-3},
        {'params': loss_fn.parameters(), 'lr': 1e-2},
    ],
    weight_decay=5e-4,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

print(f'ArcFaceLoss configurée : num_classes={NUM_CLASSES}, embed_dim={EMBED_DIM}')
print(f'Prototypes de classe (W) : {sum(p.numel() for p in loss_fn.parameters()):,} paramètres')
print(f'Batches train : {len(train_loader)} | Batches val : {len(val_loader)}')


In [ ]:
best_recall = 0
patience    = 10
no_improve  = 0
NUM_EPOCHS  = 20

best_model_path = os.path.join(SAVE_DIR, 'MobileNetV2_finetuned.pth')

# Historique pour les courbes
history = {
    'epoch':        [],
    'train_loss':   [],
    'val_loss':     [],
    'val_recall@1': [],
    'lr':           [],
}

for epoch in range(NUM_EPOCHS):
    # ============ TRAIN ============
    model.train()
    train_loss_sum = 0.0
    n_train_batches = 0

    for images, labels_batch in tqdm(train_loader, desc=f'Epoch {epoch+1} [train]'):
        images       = images.to(device)
        labels_batch = labels_batch.to(device)

        # Pas de normalisation manuelle : ArcFaceLoss normalise W et
        # les embeddings en interne pour calculer le cosinus.
        embeddings = model(images)
        loss       = loss_fn(embeddings, labels_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item()
        n_train_batches += 1

    scheduler.step()
    avg_train_loss = train_loss_sum / max(n_train_batches, 1)

    # ============ VALIDATION ============
    model.eval()
    val_loss_sum = 0.0
    n_val_batches = 0
    all_embs, all_lbls = [], []

    with torch.no_grad():
        for images, lbls in tqdm(val_loader, desc=f'Epoch {epoch+1} [val]  '):
            images = images.to(device)
            lbls_t = lbls.to(device)

            embeddings = model(images)
            # Loss validation : mêmes prototypes W mais sans backward
            v_loss = loss_fn(embeddings, lbls_t)
            val_loss_sum += v_loss.item()
            n_val_batches += 1

            # Pour le Recall on prend les embeddings normalisés
            emb_n = F.normalize(embeddings, p=2, dim=1)
            all_embs.append(emb_n.cpu())
            all_lbls.extend(lbls.tolist())

    avg_val_loss = val_loss_sum / max(n_val_batches, 1)

    # Recall@1 sur le set de validation (query = chaque image val,
    # gallery = autres images val de même split)
    all_embs = torch.cat(all_embs)
    sim      = torch.mm(all_embs, all_embs.T)

    correct = 0
    for i in range(len(all_embs)):
        scores     = sim[i].clone()
        scores[i]  = -1.0
        top1_label = all_lbls[torch.argmax(scores).item()]
        if top1_label == all_lbls[i]:
            correct += 1

    val_recall_at_1 = correct / len(all_embs) * 100
    current_lr      = optimizer.param_groups[0]['lr']

    
    history['epoch'].append(epoch + 1)
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_recall@1'].append(val_recall_at_1)
    history['lr'].append(current_lr)

    print(f'Epoch {epoch+1:>3} | '
          f'Train Loss {avg_train_loss:.4f} | '
          f'Val Loss {avg_val_loss:.4f} | '
          f'Val Recall@1 {val_recall_at_1:.1f}% | '
          f'LR {current_lr:.2e}')

    # ============ CHECKPOINT + EARLY STOPPING ============
    if val_recall_at_1 > best_recall:
        best_recall = val_recall_at_1
        torch.save({
            'model_state':    model.state_dict(),
            'loss_state':     loss_fn.state_dict(),  # on sauve aussi W au cas où
            'label2idx':      full_train_ds.label2idx,
            'embed_dim':      EMBED_DIM,
            'best_recall@1':  val_recall_at_1,
            'epoch':          epoch + 1,
        }, best_model_path)
        no_improve  = 0
        print(f'  ✓ Nouveau meilleur modèle sauvegardé (Val Recall@1 = {val_recall_at_1:.1f}%)')
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"\nEarly stopping à l'epoch {epoch+1}")
            break

# Sauvegarde de l'historique en JSON pour réutilisation ultérieure
history_path = os.path.join(SAVE_DIR, 'training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nMeilleur Val Recall@1 : {best_recall:.1f}%')
print(f'Modèle sauvegardé   : {best_model_path}')
print(f'Historique sauvé    : {history_path}')


In [ ]:
# Charge l'historique (utile aussi pour re-générer le graphe sans relancer l'entraînement)
with open(os.path.join(SAVE_DIR, 'training_history.json')) as f:
    history = json.load(f)

epochs       = history['epoch']
train_loss   = history['train_loss']
val_loss     = history['val_loss']
val_recall   = history['val_recall@1']

# Best epoch (max recall@1)
best_idx     = max(range(len(val_recall)), key=lambda i: val_recall[i])
best_epoch   = epochs[best_idx]
best_recall  = val_recall[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Sous-graphe 1 : Loss train vs val ---
ax = axes[0]
ax.plot(epochs, train_loss, marker='o', linewidth=2, markersize=6,
        color='#1f77b4', label='Train Loss')
ax.plot(epochs, val_loss,   marker='s', linewidth=2, markersize=6,
        color='#d62728', label='Validation Loss')
ax.axvline(best_epoch, color='gray', linestyle=':', alpha=0.7,
           label=f'Best epoch ({best_epoch})')
ax.set_title('Courbes de Loss — ArcFace', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.6)
ax.set_xticks(epochs)

# --- Sous-graphe 2 : Recall@1 validation ---
ax = axes[1]
ax.plot(epochs, val_recall, marker='D', linewidth=2, markersize=6,
        color='#2ca02c', label='Val Recall@1')
ax.axvline(best_epoch, color='gray', linestyle=':', alpha=0.7)
ax.axhline(best_recall, color='gray', linestyle=':', alpha=0.5)
ax.annotate(f'Best : {best_recall:.1f}% @ epoch {best_epoch}',
            xy=(best_epoch, best_recall),
            xytext=(10, -20), textcoords='offset points',
            fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7),
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_title('Recall@1 sur le set de validation', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Recall@1 (%)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.6)
ax.set_xticks(epochs)
ax.set_ylim(0, 105)

plt.tight_layout()

curves_path = os.path.join(SAVE_DIR, 'training_curves_ArcFace.png')
plt.savefig(curves_path, dpi=300, bbox_inches='tight')
print(f'Courbes sauvegardées : {curves_path}')
plt.show()

# Résumé textuel
print('\n--- Résumé de l\'entraînement ---')
print(f'Epochs réalisés   : {len(epochs)}')
print(f'Train loss final  : {train_loss[-1]:.4f}')
print(f'Val loss final    : {val_loss[-1]:.4f}')
print(f'Meilleur Recall@1 : {best_recall:.1f}% (epoch {best_epoch})')
print(f'Train loss min    : {min(train_loss):.4f}')
print(f'Val loss min      : {min(val_loss):.4f}')


## 3. Extraction et évaluation après finetuning ArcFace

In [ ]:
checkpoint = torch.load(os.path.join(SAVE_DIR, 'MobileNetV2_finetuned.pth'))
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f"Modèle finetuné chargé (best Recall@1 = {checkpoint['best_recall@1']:.1f}%).")

In [ ]:
# On réutilise le dataset sans augmentation (transform_eval)
dataset_eval    = SimpleImageDataset(root_dir=IMAGE_DIR, transform=transform_eval)
dataloader_eval = DataLoader(dataset_eval, batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=4)

all_embeddings = []
all_filenames  = []

print(f'Extraction des vecteurs pour {len(dataset_eval)} images...')

with torch.no_grad():
    for images, filenames in tqdm(dataloader_eval):
        images = images.to(device)
        embeddings = model(images)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
        all_filenames.extend(filenames)

database_tensor = torch.cat(all_embeddings, dim=0)

save_path_ft = os.path.join(SAVE_DIR, 'MobileNetV2_finetuned_best.pth')
torch.save({'embeddings': database_tensor, 'filenames': all_filenames}, save_path_ft)

print(f'Base de données créée : {database_tensor.shape}')
print(f'Sauvegardée : {save_path_ft}')

In [ ]:
db         = torch.load(os.path.join(SAVE_DIR, 'MobileNetV2_finetuned_best.pth'))
embeddings = db['embeddings']
filenames  = db['filenames']
labels     = [get_label_from_filename(f) for f in filenames]

K_values    = [1, 5, 20, 50]
recalls     = {k: 0 for k in K_values}
ap_scores   = []
num_queries = len(embeddings)

print(f'Évaluation du Recall sur {num_queries} images...')

similarity_matrix = torch.mm(embeddings, embeddings.T)

for i in range(num_queries):
    query_label = labels[i]
    scores      = similarity_matrix[i].clone()
    scores[i]   = -1.0

    _, top_indices   = torch.topk(scores, max(K_values))
    retrieved_labels = [labels[idx.item()] for idx in top_indices]

    for k in K_values:
        if query_label in retrieved_labels[:k]:
            recalls[k] += 1

    num_relevant = labels.count(query_label) - 1
    if num_relevant == 0:
        continue

    hits, precision_sum = 0, 0.0
    for rank, lbl in enumerate(retrieved_labels, start=1):
        if lbl == query_label:
            hits += 1
            precision_sum += hits / rank
    ap_scores.append(precision_sum / num_relevant)

recall_scores = {k: recalls[k] / num_queries * 100 for k in K_values}
mAP           = sum(ap_scores) / len(ap_scores) * 100

print('\n--- Résultats MobileNetV2 ArcFace Finetuned ---')
for k, v in recall_scores.items():
    print(f'Recall@{k:>2} : {v:.2f}%')
print(f'mAP        : {mAP:.2f}%')


plt.figure(figsize=(10, 6))
plt.plot(K_values, list(recall_scores.values()),
         marker='D', linestyle='-', color='#1f77b4',
         linewidth=2.5, markersize=8, label='MobileNetV2 ArcFace')

for k, v in recall_scores.items():
    plt.annotate(f'{v:.1f}%', (k, v),
                 textcoords='offset points',
                 xytext=(0, 10), ha='center', fontsize=10)

plt.text(0.98, 0.05, f'mAP : {mAP:.2f}%',
         transform=plt.gca().transAxes, fontsize=11,
         ha='right', va='bottom',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.title('Courbe de Rappel (Recall@K) - MobileNetV2 ArcFace Finetuned',
          fontsize=14, fontweight='bold')
plt.xlabel('Rang K (Nombre de résultats affichés)', fontsize=12)
plt.ylabel('Taux de Rappel (%)', fontsize=12)
plt.xticks(K_values)
plt.ylim(0, 105)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()

graph_filename = 'recall_curve_MobileNetV2_ArcFace'
graph_path     = os.path.join(SAVE_DIR, f'{graph_filename}.png')
result_path    = os.path.join(SAVE_DIR, f'results_{graph_filename}.json')

plt.savefig(graph_path, dpi=300, bbox_inches='tight')

results = {
    'model':    graph_filename,
    'recall':   {f'@{k}': round(v, 2) for k, v in recall_scores.items()},
    'mAP':      round(mAP, 2),
    'n_images': num_queries,
}
with open(result_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Graphique sauvegardé : {graph_path}')
print(f'Résultats sauvegardés : {result_path}')
plt.show()

## 4. Comparaison Zero-Shot vs ArcFace Finetuned

In [ ]:
with open(os.path.join(SAVE_DIR, 'results_recall_curve_MobileNetV2_zero_shot.json')) as f:
    zero_shot = json.load(f)

with open(os.path.join(SAVE_DIR, 'results_recall_curve_MobileNetV2_ArcFace.json')) as f:
    arcface = json.load(f)

K_values       = [1, 5, 20, 50]
scores_zero    = [zero_shot['recall'][f'@{k}'] for k in K_values]
scores_arcface = [arcface['recall'][f'@{k}']    for k in K_values]

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(K_values, scores_zero, marker='o', linewidth=2.5, markersize=8,
        color='#ff7f0e',
        label=f"Zero-Shot (mAP {zero_shot['mAP']}%)")
ax.plot(K_values, scores_arcface, marker='D', linewidth=2.5, markersize=8,
        color='#1f77b4',
        label=f"ArcFace Finetuned (mAP {arcface['mAP']}%)")

for k, v in zip(K_values, scores_zero):
    ax.annotate(f'{v:.1f}%', (k, v), textcoords='offset points',
                xytext=(-15, 6), fontsize=9)
for k, v in zip(K_values, scores_arcface):
    ax.annotate(f'{v:.1f}%', (k, v), textcoords='offset points',
                xytext=(5, 6), fontsize=9)

ax.set_title('Recall@K — MobileNetV2 Zero-Shot vs ArcFace Finetuned',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Rang K (Nombre de résultats affichés)', fontsize=12)
ax.set_ylabel('Taux de Rappel (%)', fontsize=12)
ax.set_xticks(K_values)
ax.set_ylim(0, 105)
ax.grid(True, linestyle='--', alpha=0.7)
ax.legend(fontsize=11)

plt.tight_layout()

save_path = os.path.join(SAVE_DIR, 'comparaison_recall_MobileNetV2_ArcFace.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f'Comparaison sauvegardée : {save_path}')
plt.show()